# 05. Підбір параметрів через Optuna (лекція 19)

**Тиждень 3.** Підбираємо параметри класичного пайплайну (`DEFAULT_PARAMS` у
`src/pipeline.py`), щоб максимізувати decode rate.

**Важливо:** підбираємо на **train** (одні коди), перевіряємо на **test** (інші коди,
інший seed). Інакше параметри «підженуться» під тест, і результат буде нечесним
(overfitting, лекція 15).

In [ ]:
import sys
from pathlib import Path

# ноутбук лежить у notebooks/, корінь проєкту на рівень вище
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

In [ ]:
import optuna

from src.data import generate_synthetic
from src.decoders import DECODERS
from src.degradations import DEGRADATIONS
from src.evaluate import check
from src.pipeline import DEFAULT_PARAMS, classic_pipeline

TUNE_DEGRADATIONS = ["dark", "uneven", "glare", "defocus", "noise"]
DECODER = "zxing"

train_samples = generate_synthetic(n_per_format=5, seed=100)
test_samples = generate_synthetic(n_per_format=10, seed=200)

## Train-кадри: кожен код, кожне пошкодження, рівень 3

In [ ]:
import itertools

train_images = [(DEGRADATIONS[name](sample["image"], 3, seed=7), sample["text"])
                for sample, name in itertools.product(train_samples, TUNE_DEGRADATIONS)]
print("Train-кадрів:", len(train_images))

In [ ]:
def objective(trial):
    """Optuna викликає цю функцію з різними параметрами і максимізує результат."""
    params = {
        "flat_kernel_part": trial.suggest_float("flat_kernel_part", 0.03, 0.2),
        "clahe_clip": trial.suggest_float("clahe_clip", 0.5, 6.0),
        "clahe_tile": trial.suggest_int("clahe_tile", 2, 16),
        "unsharp_amount": trial.suggest_float("unsharp_amount", 0.0, 3.0),
    }
    outcomes = [check(DECODERS[DECODER](classic_pipeline(image, params)), text)
                for image, text in train_images]
    return np.mean([outcome == "correct" for outcome in outcomes])


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=30)   # TODO: більше спроб = краще, але довше
print("Найкращий train decode rate:", round(study.best_value, 3))
print("Параметри:", study.best_params)

## Перевірка на test: стандартні параметри проти підібраних

In [ ]:
rows = []
param_sets = {"default": DEFAULT_PARAMS, "tuned": study.best_params}
for params_name, sample, name, level in itertools.product(param_sets, test_samples,
                                                          TUNE_DEGRADATIONS, [1, 2, 3, 4, 5]):
    damaged = DEGRADATIONS[name](sample["image"], level, seed=level)
    results = DECODERS[DECODER](classic_pipeline(damaged, param_sets[params_name]))
    rows.append({"params": params_name, "degradation": name, "level": level,
                 "correct": check(results, sample["text"]) == "correct"})
tuning = pd.DataFrame(rows)
tuning.to_csv(RESULTS / "05_optuna.csv", index=False)
tuning.pivot_table(index="degradation", columns="params", values="correct", aggfunc="mean").round(3)

**TODO для тебе:**
- Чи переноситься приріст з train на test?
- Спробуй підбирати параметри `adaptive` (пороги `THRESHOLDS`) замість `classic`.
- Що буде, якщо підбирати на одному типі пошкодження, а тестувати на іншому?